# Bigram Language Model (Character-Level)

Das ist der einfachste mögliche Baustein eines LLMs. Wir sehen hier einmal den **kompletten Trainings-Loop**:

Daten laden → Tokenizer → Modell → Loss → Backprop → Sampling

Das Bigram-Modell sagt das nächste Zeichen **nur basierend auf dem aktuellen Zeichen** voraus (kein Kontext, keine Attention). Es ist im Grunde nur eine Lookup-Tabelle. Trotzdem lernen wir daran alle Bausteine, die wir später für einen echten Transformer brauchen.

Trainingsdaten: Tiny Shakespeare Korpus (~1MB reiner Text).

## 1. Imports und Setup

`torch.manual_seed` sorgt dafür, dass die Zufallszahlen reproduzierbar sind – wichtig, um Ergebnisse vergleichbar zu machen, während du experimentierst.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Verwende device: {device}")

## 2. Trainingsdaten laden

Die Datei `data/input.txt` enthält den kompletten Shakespeare-Korpus als reinen Text.

In [ ]:
with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Länge des Textes in Zeichen: {len(text)}")
print("---")
print(text[:300])

## 3. Vokabular untersuchen

Beim Character-Level-Modell besteht unser "Vokabular" aus allen einzigartigen Zeichen im Text (Buchstaben, Satzzeichen, Leerzeichen, Zeilenumbrüche). Das ist viel kleiner als ein Wort- oder Subword-Vokabular (typischerweise nur ~65 Zeichen statt zehntausender Tokens), macht aber auch das Modell weniger effizient beim Lernen von Bedeutung.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vokabulargröße: {vocab_size}")
print(f"Zeichen: {''.join(chars)}")

## 4. Tokenizer bauen

Ganz simpel: jedes Zeichen bekommt eine eindeutige Zahl (Integer) zugewiesen.
- `encode`: String → Liste von Zahlen
- `decode`: Liste von Zahlen → String

Echte LLMs nutzen hier meist **BPE (Byte Pair Encoding)** auf Subword-Ebene (z.B. tiktoken bei GPT), aber character-level ist der einfachste Einstieg, um das Prinzip zu verstehen.

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}  # string to index
itos = {i: ch for i, ch in enumerate(chars)}  # index to string

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# Kurzer Test
test = encode("Hallo Welt")
print(f"Encoded: {test}")
print(f"Decoded: {decode(test)}")

## 5. Kompletten Text encodieren und in Train/Val splitten

Wir wandeln den gesamten Text in einen Tensor aus Zahlen um. Danach splitten wir in Trainings- (90%) und Validierungsdaten (10%), um später zu prüfen, ob das Modell überfittet (auswendig lernt statt zu generalisieren).

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Shape: {data.shape}, dtype: {data.dtype}")

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train: {len(train_data)} Zeichen, Val: {len(val_data)} Zeichen")

## 6. Batches erzeugen

Wir trainieren nicht auf dem kompletten Text auf einmal, sondern auf zufälligen kleinen Ausschnitten (Chunks).

- `block_size`: wie viele Zeichen Kontext das Modell sieht (hier erstmal egal, da Bigram nur 1 Zeichen zurückschaut, aber wir bauen es allgemein, damit wir später leicht auf Transformer umsteigen können)
- `batch_size`: wie viele solcher Chunks wir parallel verarbeiten (für effizientes GPU/CPU-Training)

`x` ist der Input, `y` ist der Input um eins nach rechts verschoben (das, was vorhergesagt werden soll).

In [ ]:
block_size = 8
batch_size = 4

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print("Inputs (x):")
print(xb)
print("Targets (y):")
print(yb)

## 7. Das Bigram-Modell

Das Modell besteht nur aus einer **Embedding-Tabelle** der Größe `vocab_size x vocab_size`. Für jedes Eingabe-Zeichen schlägt es direkt eine Zeile nach, die die "Logits" (unnormalisierte Wahrscheinlichkeiten) für das nächste Zeichen enthält.

Kein Attention, keine Hidden Layers – reines Nachschlagen. Genau deswegen ist es der perfekte Startpunkt: du siehst den Trainings-Mechanismus, ohne von Architektur-Komplexität abgelenkt zu werden.

In [ ]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Jede Zeile i enthält die Logits fürs nächste Zeichen, wenn Zeichen i das aktuelle ist
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx und targets sind (B, T) Tensoren aus Integers
        logits = self.token_embedding_table(idx)  # (B, T, C) - C = vocab_size

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx ist (B, T) - der aktuelle Kontext
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]  # nur letztes Zeitschritt betrachten -> (B, C)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size).to(device)
logits, loss = model(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss.item():.4f}  (Erwartungswert bei Zufall: {torch.log(torch.tensor(vocab_size)).item():.4f})")

**Zur Loss-Referenz:** Bei komplett zufälligen Vorhersagen erwarten wir einen Loss von `-ln(1/vocab_size)`. Ist unser aktueller Loss ungefähr in dieser Größenordnung? Das ist normal, bevor trainiert wurde – das Modell rät noch komplett zufällig.

## 8. Text generieren VOR dem Training

Nur um zu sehen, wie kompletter Kauderwelsch aussieht, bevor überhaupt trainiert wurde.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)  # Start-Token: Zeichen 0
generated = model.generate(context, max_new_tokens=100)
print(decode(generated[0].tolist()))

## 9. Training-Loop

Wir nutzen den **AdamW-Optimizer** (Standard für Transformer-Training) und trainieren über mehrere Steps. Jeder Step:
1. Batch ziehen
2. Forward Pass (Logits + Loss berechnen)
3. Gradienten zurücksetzen
4. Backward Pass (Gradienten berechnen)
5. Optimizer-Step (Gewichte updaten)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 32
max_iters = 10000
eval_interval = 1000

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"\nFinaler Loss: {loss.item():.4f}")

## 10. Text generieren NACH dem Training

Immer noch kein "echter" Shakespeare (dafür fehlt dem Modell jeglicher Kontext über einzelne Zeichen hinaus), aber du solltest sehen, dass der Loss deutlich gesunken ist und der generierte Text zumindest Wort-ähnliche Strukturen und Groß-/Kleinschreibungs-Muster zeigt.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))

## 11. Nächste Schritte

Das Bigram-Modell hat eine fundamentale Grenze: es schaut **nur ein Zeichen zurück**. Es kann keine längeren Muster, Wörter oder Grammatik lernen.

Die nächsten Bausteine, die wir daraus einen echten (Mini-)Transformer machen:

1. **Self-Attention**: jedes Zeichen "schaut" auf vorherige Zeichen im Kontext, statt nur auf das direkt vorherige
2. **Positional Embeddings**: dem Modell mitteilen, an welcher Position im Kontext ein Zeichen steht
3. **Multi-Head Attention**: mehrere Attention-Mechanismen parallel, die unterschiedliche Muster lernen
4. **Feed-Forward-Layer + Residual Connections + LayerNorm**: die restlichen Bausteine eines Transformer-Blocks
5. **Mehrere Transformer-Blöcke stapeln**: die eigentliche "Tiefe" des Modells

Jeder dieser Punkte lässt sich einzeln in dieses Notebook einbauen, Schritt für Schritt – sag Bescheid, wenn du bereit für Self-Attention bist.